In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_delivery_eda.csv"
REPORT_PATH = PROJECT_ROOT / "reports" / "model_results.csv"

In [5]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Missing values:\n", df.isna().sum())
df.head()

Shape: (41953, 15)
Missing values:
 ID                               0
Restaurant_latitude              0
Restaurant_longitude             0
Delivery_location_latitude       0
Delivery_location_longitude      0
Weatherconditions              569
Road_traffic_density             0
Type_of_vehicle                  0
Time_taken(min)                  0
distance_km                      0
pickup_hour                      0
pickup_minute                    0
is_peak_hour                     0
Traffic_Score                    0
Distance_Traffic                 0
dtype: int64


,ID,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Weatherconditions,Road_traffic_density,Type_of_vehicle,Time_taken(min),distance_km,pickup_hour,pickup_minute,is_peak_hour,Traffic_Score,Distance_Traffic
0,0x4607,22.745049,75.892471,22.765049,75.912471,Sunny,High,motorcycle,24,3.025149,11,45,0,3,9.075448
1,0xb379,12.913041,77.683237,13.043041,77.813237,Stormy,Jam,scooter,33,20.183530,19,50,1,4,80.734119
2,0x5d6d,12.914264,77.678400,12.924264,77.688400,Sandstorms,Low,motorcycle,26,1.552758,8,45,0,1,1.552758
3,0x7a6a,11.003669,76.976494,11.053669,77.026494,Sunny,Medium,motorcycle,21,7.790401,18,10,0,2,15.580803
4,0x70a2,12.972793,80.249982,13.012793,80.289982,Cloudy,High,scooter,30,6.210138,13,45,1,3,18.630414


//Step 1 — Séparer X et y

In [9]:
# Étape 3.1 — Séparer les variables explicatives et la cible
TARGET = "Time_taken(min)"
X = df.drop(columns=[TARGET, "ID"], errors="ignore")
y = df[TARGET]

print("X shape :", X.shape)
print("y shape :", y.shape)
print("Features :", X.columns.tolist())
print("Target :", y.name)

X shape : (41953, 13)
y shape : (41953,)
Features : ['Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Weatherconditions', 'Road_traffic_density', 'Type_of_vehicle', 'distance_km', 'pickup_hour', 'pickup_minute', 'is_peak_hour', 'Traffic_Score', 'Distance_Traffic']
Target : Time_taken(min)


In [4]:
# 2. Division des données (80% entraînement, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Taille de l entraînement : {X_train.shape}')
print(f'Taille du test : {X_test.shape}')

Taille de l entraînement : (33562, 13)
Taille du test : (8391, 13)


In [6]:
# Étape 3.2 — Encoder et standardiser dans un pipeline
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "str", "category"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

print("Variables numériques :", numeric_features)
print("Variables catégorielles :", categorical_features)

Variables numériques : ['Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'distance_km', 'pickup_hour', 'pickup_minute', 'is_peak_hour', 'Traffic_Score', 'Distance_Traffic']
Variables catégorielles : ['Weatherconditions', 'Road_traffic_density', 'Type_of_vehicle']


In [7]:
# Étape 3.3 — Définir quatre modèles de régression
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=150, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

print("Modèles à comparer :", list(models))

Modèles à comparer : ['Linear Regression', 'Ridge', 'Random Forest', 'Gradient Boosting']


In [8]:
# Étape 3.4 — Entraîner et évaluer les modèles
results = []
trained_pipelines = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    trained_pipelines[name] = pipeline

    mse = mean_squared_error(y_test, y_pred)
    results.append({
        "Modèle": name,
        "MAE (min)": mean_absolute_error(y_test, y_pred),
        "MSE": mse,
        "RMSE (min)": np.sqrt(mse),
        "R²": r2_score(y_test, y_pred),
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("RMSE (min)")
    .reset_index(drop=True)
)

results_df.to_csv(REPORT_PATH, index=False)
print(results_df.round(4).to_string(index=False))
print(f"\nRésultats sauvegardés dans : {REPORT_PATH}")

           Modèle  MAE (min)     MSE  RMSE (min)     R²
Gradient Boosting     5.3884 46.1405      6.7927 0.4645
    Random Forest     5.5352 50.2251      7.0870 0.4170
            Ridge     6.3565 62.8142      7.9255 0.2709
Linear Regression     6.3565 62.8143      7.9255 0.2709

Résultats sauvegardés dans : /home/mouad/DeliveryETA _Livraison_2eme_brief/reports/model_results.csv


## Étape 4 — Choisir et affiner le meilleur modèle

Le Gradient Boosting est retenu comme meilleur candidat selon le RMSE obtenu à l'étape précédente. Ses hyperparamètres sont optimisés avec une recherche aléatoire, puis ses performances sont comparées entre l'entraînement et le test.

In [10]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

search_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(random_state=42)),
])

param_distributions = {
    "model__n_estimators": randint(80, 251),
    "model__learning_rate": uniform(0.02, 0.18),
    "model__max_depth": randint(2, 6),
    "model__min_samples_split": randint(2, 11),
    "model__min_samples_leaf": randint(1, 6),
    "model__subsample": uniform(0.7, 0.3),
}

random_search = RandomizedSearchCV(
    estimator=search_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=42,
    n_jobs=-1,
    refit=True,
)

random_search.fit(X_train, y_train)
final_pipeline = random_search.best_estimator_

print("Meilleurs hyperparamètres:")
print(random_search.best_params_)
print(f"Meilleur RMSE CV: {-random_search.best_score_:.4f} minutes")

Meilleurs hyperparamètres:
{'model__learning_rate': np.float64(0.028399819378450777), 'model__max_depth': 5, 'model__min_samples_leaf': 3, 'model__min_samples_split': 8, 'model__n_estimators': 143, 'model__subsample': np.float64(0.8400288679743939)}
Meilleur RMSE CV: 6.7720 minutes


In [ ]:
from joblib import dump

# Évaluer le modèle final sur train et test
X_train_transformed = final_pipeline.named_steps["preprocessor"].transform(X_train)
n_features = X_train_transformed.shape[1]

def adjusted_r2(r2, n, p):
    if n <= p + 1:
        return np.nan
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

def evaluate_split(split_name, features, target_values):
    predictions = final_pipeline.predict(features)
    mse = mean_squared_error(target_values, predictions)
    r2 = r2_score(target_values, predictions)
    return {
        "Ensemble": split_name,
        "MAE (min)": mean_absolute_error(target_values, predictions),
        "RMSE (min)": np.sqrt(mse),
        "R²": r2,
        "R² ajusté": adjusted_r2(r2, len(target_values), n_features),
    }

final_metrics = pd.DataFrame([
    evaluate_split("Entraînement", X_train, y_train),
    evaluate_split("Test", X_test, y_test),
]).round(4)

final_metrics

,Ensemble,MAE (min),RMSE (min),R²,R² ajusté
0,Entraînement,5.2625,6.6719,0.4968,0.4964
1,Test,5.2770,6.6741,0.4830,0.4815


In [12]:
# Sauvegarder les métriques et les artefacts pour l'inférence
final_metrics.to_csv(PROJECT_ROOT / "reports" / "final_model_metrics.csv", index=False)
dump(final_pipeline, PROJECT_ROOT / "models" / "final_delivery_model.joblib")
dump(final_pipeline.named_steps["preprocessor"], PROJECT_ROOT / "models" / "delivery_preprocessor.joblib")

train_mae = final_metrics.loc[final_metrics["Ensemble"] == "Entraînement", "MAE (min)"].iloc[0]
test_mae = final_metrics.loc[final_metrics["Ensemble"] == "Test", "MAE (min)"].iloc[0]
train_r2 = final_metrics.loc[final_metrics["Ensemble"] == "Entraînement", "R²"].iloc[0]
test_r2 = final_metrics.loc[final_metrics["Ensemble"] == "Test", "R²"].iloc[0]

print(
    f"Métier: le modèle se trompe en moyenne de {test_mae:.2f} minutes "
    "sur des commandes jamais vues."
)
print(f"Écart MAE train/test: {abs(train_mae - test_mae):.2f} minute(s)")
print(f"Écart R² train/test: {abs(train_r2 - test_r2):.4f}")
if abs(train_r2 - test_r2) < 0.05:
    print("Diagnostic: pas de signe important de surapprentissage.")
else:
    print("Diagnostic: écart train/test à surveiller pour un possible surapprentissage.")
print("Modèle sauvegardé dans models/final_delivery_model.joblib")

Métier: le modèle se trompe en moyenne de 5.28 minutes sur des commandes jamais vues.
Écart MAE train/test: 0.01 minute(s)
Écart R² train/test: 0.0138
Diagnostic: pas de signe important de surapprentissage.
Modèle sauvegardé dans models/final_delivery_model.joblib
